## **Extract, Transform, Load (ETL) Process for SDW2023 User Data**

### Extract: Fetching User Data from SDW2023 API

In [ ]:
!pip install pandas

In [3]:
sdw2023_url = 'http://localhost:8090'

In [4]:
import pandas as pd

df = pd.read_csv('SDW2023.csv')
user_ids = df['UserID'].tolist()

print(user_ids)

[1, 100, 101, 102]


In [5]:
import requests
import json


def get_user(id):
    response = requests.get(f'{sdw2023_url}/users/{id}')
    return response.json() if response.status_code == 200 else None


users = [user for user_id in user_ids if (user := get_user(user_id)) is not None]
print(json.dumps(users, indent=2, ensure_ascii=False))

[
  {
    "id": 100,
    "name": "Devweekerson",
    "account": {
      "id": 100,
      "number": "01.097954-4",
      "agency": "2030",
      "balance": 624.12,
      "limit": 1000.0
    },
    "card": {
      "id": 100,
      "number": "xxxx xxxx xxxx 1111",
      "limit": 2000.0
    },
    "features": [
      {
        "id": 100,
        "icon": "https://digitalinnovationone.github.io/santander-dev-week-2023-api/icons/pix.svg",
        "description": "PIX"
      },
      {
        "id": 101,
        "icon": "https://digitalinnovationone.github.io/santander-dev-week-2023-api/icons/pay.svg",
        "description": "Pagar"
      },
      {
        "id": 102,
        "icon": "https://digitalinnovationone.github.io/santander-dev-week-2023-api/icons/transfer.svg",
        "description": "Transferir"
      },
      {
        "id": 103,
        "icon": "https://digitalinnovationone.github.io/santander-dev-week-2023-api/icons/account.svg",
        "description": "Conta Corrente"
      },
  

### Tranform: Gerando mensagens personalizadas usando OpenRouter API

#### OpenRouter APIKEY
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

#### StepFun model
stepfun/step-3.5-flash:free

In [ ]:
!pip install openai

In [6]:
openrouter_api_key = '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'

In [8]:
from openai import OpenAI

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=openrouter_api_key
)


# noinspection PyTypeChecker
def generate_ai_news(user):

    response = client.chat.completions.create(
        model="stepfun/step-3.5-flash:free",
        messages=[
            {
                "role": "system",
                "content": "Você é um especialista em marketing bancário."
            },
            {
                "role": "user",
                "content": f"Crie uma mensagem curta (máx 100 caracteres) para {user['name']} sobre a importância dos investimentos."
            }
        ],
        extra_body={"reasoning": {"enabled": True}}
    )

    return response.choices[0].message.content or ""


for user in users:
    news = generate_ai_news(user)
    print(f"Mensagem para {user['name']}: {news}")
    user['news'].append({
        "icon": "https://digitalinnovationone.github.io/santander-dev-week-2023-api/icons/credit.svg",
        "description": news
    })

Mensagem para Devweekerson: Devweekerson: Invista e transforme renda em patrimônio. Planeje seu futuro!
Mensagem para Maria Silva: Maria, invista para garantir seu futuro. Faça seu dinheiro trabalhar por você.
Mensagem para João Santos: João Santos, seu dinheiro vale mais investido. Descubra como!


### LOAD: Atualizando os dados dos usuários com as mensagens geradas

In [9]:
def update_user(user):
    response = requests.put(f'{sdw2023_url}/users/{user["id"]}', json=user)
    return True if response.status_code == 200 else False

for user in users:
    if update_user(user):
        print(f"Usuário {user['name']} atualizado!")
    else:
        print(f"Falha ao atualizar o usuário {user['name']}.")

Usuário Devweekerson atualizado!
Usuário Maria Silva atualizado!
Usuário João Santos atualizado!
